# StockVision AI — LSTM Model Notebook
Exploratory training, hyperparameter search, and evaluation.
Run cells top-to-bottom after installing requirements.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from utils.data_handler import (
    fetch_stock_data, add_technical_indicators,
    prepare_lstm_data, inverse_transform_predictions
)
from models.lstm_model import (
    build_lstm_model, build_gru_model,
    train_model, evaluate_model, forecast_future
)

print('Libraries loaded ✓')

## 1. Load & Preprocess Data

In [ ]:
TICKER = 'AAPL'
df_raw = fetch_stock_data(TICKER, period='2y')
df     = add_technical_indicators(df_raw)
print(f'Shape: {df.shape}')
df.tail()

## 2. Prepare LSTM Sequences

In [ ]:
FEATURE_COLS = [
    'Open','High','Low','Volume',
    'SMA_20','SMA_50','EMA_12','EMA_26',
    'RSI_14','MACD','MACD_Signal',
    'BB_Upper','BB_Lower','BB_Width','ATR_14','Daily_Return',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
WINDOW = 60

X_tr, y_tr, X_te, y_te, sc_f, sc_t, train_size = prepare_lstm_data(
    df, feature_cols=FEATURE_COLS, window=WINDOW, test_ratio=0.2
)

val_split = int(len(X_tr) * 0.85)
X_val, y_val = X_tr[val_split:], y_tr[val_split:]
X_tr2, y_tr2 = X_tr[:val_split], y_tr[:val_split]

print(f'Train: {X_tr2.shape} | Val: {X_val.shape} | Test: {X_te.shape}')

## 3. Build & Train LSTM

In [ ]:
model = build_lstm_model(
    input_shape=(WINDOW, len(FEATURE_COLS)),
    units=[128, 64, 32],
    dropout=0.2,
)
model.summary()

history, model = train_model(
    model, X_tr2, y_tr2, X_val, y_val,
    epochs=100, batch_size=32, ticker=TICKER
)

## 4. Evaluate & Forecast

In [ ]:
metrics = evaluate_model(model, X_te, y_te, sc_t)
print('\n── Metrics ──')
for k, v in metrics.items():
    if k not in ('predictions','actual'):
        print(f'  {k:12s}: {v}')

future = forecast_future(model, X_te[-1], sc_t, n_steps=30)
print(f'\n30-day forecast (first 5): {future[:5]}')

## 5. Plot Results

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(metrics['actual'],     label='Actual',    color='#00D4FF')
plt.plot(metrics['predictions'],label='Predicted', color='#FF6B35', linestyle='--')
plt.title(f'{TICKER} — LSTM Test Predictions', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 3))
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curves')
plt.legend()
plt.tight_layout()
plt.show()

## 6. GRU Comparison

In [ ]:
gru = build_gru_model(input_shape=(WINDOW, len(FEATURE_COLS)))
_, gru = train_model(gru, X_tr2, y_tr2, X_val, y_val,
                      epochs=60, batch_size=32, ticker='GRU')
gru_m = evaluate_model(gru, X_te, y_te, sc_t)

comparison = pd.DataFrame([
    {'Model':'LSTM', **{k:v for k,v in metrics.items() if k not in ('predictions','actual')}},
    {'Model':'GRU',  **{k:v for k,v in gru_m.items()  if k not in ('predictions','actual')}},
])
print(comparison.to_string(index=False))